# Tempos aproximados de treinamento e ajuste por gerador/base

Fonte: logs de trials do Optuna em `results/from_s3/result_trials_<base>_<gerador>_<fold>.csv`.
A coluna `running_time` registra o tempo de parede (segundos) de **um trial** =
ajuste do gerador + amostragem dos dados sintéticos (medido em
`src/generators_tuning.py`, linhas 76–78, em torno de `generate_data`).

Reportamos, por base × gerador:

- tempo por trial (mediana, média, mín–máx) — representativo do **treinamento de um modelo**;
- tempo médio do trial vencedor (maior MCC), correspondente à configuração usada
  para gerar os dados sintéticos finais;
- tempo **total de ajuste de hiperparâmetros** (soma de todos os trials dos 5 folds).

Arquivos com sufixo `_error` são ignorados.

In [1]:
from pathlib import Path

import pandas as pd

import compute_sy_distributions as csd

pd.set_option("display.width", 200)

times_per_fold, times_summary = csd.compute_times()
times_per_fold.to_csv(csd.OUT_DIR / "training_times_per_fold.csv", index=False)
times_summary.to_csv(csd.OUT_DIR / "training_times_summary.csv", index=False)
print(f"CSVs salvos em {csd.OUT_DIR}")

CSVs salvos em /Users/felipebfg/Documents/Msc_Final_copy/review_analysis/outputs


## Resumo por base × gerador

Tempos por trial em **minutos**; total de tuning em **horas**.

In [2]:
view = times_summary.copy()
for c in ["trial_time_mean_s", "trial_time_median_s", "trial_time_min_s",
          "trial_time_max_s", "best_trial_time_mean_s"]:
    view[c.replace("_s", "_min")] = (view[c] / 60).round(1)
view = view[[
    "dataset", "generator", "folds", "n_trials_total",
    "trial_time_median_min", "trial_time_mean_min",
    "trial_time_min_min", "trial_time_max_min",
    "best_trial_time_mean_min", "total_tuning_time_h",
]]
view["total_tuning_time_h"] = view["total_tuning_time_h"].round(1)
view

,dataset,generator,folds,n_trials_total,trial_time_median_min,trial_time_mean_min,trial_time_min_min,trial_time_max_min,best_trial_time_mean_min,total_tuning_time_h
0,adult,arf,5,90,9.8,11.0,1.5,39.9,12.6,14.9
1,adult,ctabgan,5,96,90.8,94.4,83.0,176.1,89.0,151.1
2,adult,ctgan,5,100,61.1,69.4,21.0,296.7,57.3,115.6
3,adult,ddpm,5,100,44.9,47.4,24.8,99.0,45.9,79.0
4,adult,realtabformer,5,25,37.4,43.6,23.3,63.3,40.6,18.2
5,adult,tvae,5,100,21.4,26.1,11.0,74.4,22.0,43.5
6,bank_marketing,arf,5,90,4.8,6.0,1.0,14.7,7.0,9.0
7,bank_marketing,ctabgan,5,81,105.5,104.0,42.8,189.3,104.5,113.4
8,bank_marketing,ctgan,5,100,53.2,57.3,20.7,163.9,59.1,95.5
9,bank_marketing,ddpm,5,100,60.2,54.2,22.5,274.8,75.6,90.3


## Tabela por trial

Uma linha por trial do Optuna (base x gerador x fold x trial), com `running_time`
em segundos e minutos e o MCC do trial. Tabela completa salva em
`outputs/training_times_per_trial.csv`; abaixo, exibicao completa (limite de
linhas do pandas desativado apenas para esta celula).

In [3]:
times_per_trial = csd.compute_times_per_trial()
times_per_trial.to_csv(csd.OUT_DIR / "training_times_per_trial.csv", index=False)
print(f"{len(times_per_trial)} trials")

with pd.option_context("display.max_rows", None):
    display(times_per_trial.round({"running_time_s": 1, "running_time_min": 2, "MCC": 4})
            .reset_index(drop=True))

2174 trials


,dataset,generator,fold,trial,running_time_s,running_time_min,MCC
0,adult,arf,0,0,1243.2,20.72,0.5617
1,adult,arf,0,1,679.7,11.33,0.5626
2,adult,arf,0,2,525.6,8.76,0.5733
3,adult,arf,0,3,2395.6,39.93,0.5531
4,adult,arf,0,4,381.8,6.36,0.5676
5,adult,arf,0,5,742.4,12.37,0.5712
6,adult,arf,0,6,1786.6,29.78,0.5707
7,adult,arf,0,7,1618.1,26.97,0.5755
8,adult,arf,0,8,1594.6,26.58,0.5854
9,adult,arf,0,9,1079.0,17.98,0.5784


## Intervalos representativos (texto para o artigo)

Faixa entre a mediana mais baixa e a mais alta do tempo por trial entre as bases,
por gerador — útil para citar "intervalos representativos" na resposta ao revisor.

In [4]:
rng = (
    times_summary.assign(med_min=times_summary["trial_time_median_s"] / 60)
    .groupby("generator")["med_min"]
    .agg(["min", "max"])
    .round(1)
    .rename(columns={"min": "mediana_min (base mais rápida)",
                     "max": "mediana_min (base mais lenta)"})
)
total = times_summary.groupby("generator")["total_tuning_time_h"].sum().round(1)
rng["tuning_total_h (todas as bases)"] = total
rng

,mediana_min (base mais rápida),mediana_min (base mais lenta),tuning_total_h (todas as bases)
generator,,,
arf,0.3,9.8,25.9
ctabgan,5.8,105.5,283.6
ctgan,4.5,61.1,260.4
ddpm,10.2,60.2,241.3
realtabformer,8.1,37.4,66.1
tvae,3.5,21.4,94.2


In [5]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 4.5))
gens = csd.GENERATORS
x = np.arange(len(gens))
width = 0.2
for i, dataset in enumerate(csd.DATASETS):
    sub = times_summary[times_summary["dataset"] == dataset].set_index("generator").reindex(gens)
    ax.bar(x + (i - 1.5) * width, sub["trial_time_median_s"] / 60, width, label=dataset)
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(gens, rotation=20)
ax.set_ylabel("tempo mediano por trial (min, escala log)")
ax.set_title("Tempo de treinamento por trial — mediana por base e gerador")
ax.legend()
fig.tight_layout()
fig.savefig(csd.OUT_DIR / "fig_training_times.png", dpi=150)
plt.show()